# 心思维｜已有 TorchCREPE 与 VocalParse 音高交叉验证

本 Notebook 只使用 Google Drive 中已经保存的 `vocal_pitch_raw.npz`，以及仓库中已保存的 25 秒 VocalParse 测试样本。**不会运行 VocalParse、CREPE，不下载模型，也不申请 GPU。**

执行后会把转换后的参考音高轨迹和逐音符比较报告保存到 Google Drive。首次运行只会从 GitHub 下载项目源代码。

In [ ]:
from google.colab import drive
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

drive.mount('/content/drive')

# 只下载项目代码，不安装依赖、不下载模型
repo_dir = Path('/content/xinsw-music-transcriber')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'feature/vocalparse-event-musicxml',
        'https://github.com/Trifida/xinsw-music-transcriber.git', str(repo_dir)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth', '1', 'origin',
                    'feature/vocalparse-event-musicxml'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'checkout', 'FETCH_HEAD'], check=True)

def load_module(name, file_path):
    spec = importlib.util.spec_from_file_location(name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# 直接载入独立源文件，避免安装整个扒谱项目及其音频依赖
converter = load_module('xinsw_npz_converter', repo_dir / 'scripts' / 'convert_pitch_npz.py')
vocalparse = load_module('xinsw_vocalparse_adapter', repo_dir / 'musicmaster' / 'core' / 'vocalparse.py')
diagnostics = load_module('xinsw_pitch_diagnostics', repo_dir / 'musicmaster' / 'core' / 'vocalparse_diagnostics.py')

drive_root = Path('/content/drive/MyDrive/XinsW_Lead_Transcription/TorchCREPE_Test')
npz_path = drive_root / 'vocal_pitch_raw.npz'
track_json = drive_root / 'reference_pitch_track.json'
comparison_json = drive_root / 'vocalparse_pitch_comparison_30_55s.json'
vocalparse_json = repo_dir / 'tests' / 'fixtures' / 'vocalparse_25s_sample.json'

for path in (npz_path, vocalparse_json):
    if not path.is_file():
        raise FileNotFoundError(f'文件不存在：{path}')

# 已由 NPZ 检查报告确认：pitch 为 Hz，periodicity 为 0-1 指标，hop=160/16000=0.01 秒
converted = converter.convert_npz_to_pitch_json(
    npz_path, track_json,
    hop_seconds=0.01,
    frequency_key='pitch',
    confidence_key='periodicity',
)
print('参考音高轨迹转换完成')
print('帧数：', converted['frame_count'])
print('时间步长：', converted['timing']['hop_seconds'], '秒')
print('文件：', track_json)

parsed = json.loads(vocalparse_json.read_text(encoding='utf-8'))
events = vocalparse.vocalparse_tokens_to_events(parsed)
report = diagnostics.compare_vocalparse_to_pitch_track(
    events,
    parsed['bpm'],
    converted['frames'],
    confidence_threshold=0.5,
    central_fraction=0.6,
    reference_time_offset_sec=30.0,
)
comparison_json.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')

print('\\n===== 交叉验证汇总 =====')
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('\\n===== 有音高偏差的音符（按绝对半音误差排序）=====')
rows = []
for event, row in zip(events, report['events']):
    if not event['is_rest'] and row['pitch_error_semitones'] is not None:
        rows.append({
            '字': event.get('lyric') or event.get('raw_word'),
            'VocalParse MIDI': row['vocalparse_pitch_midi'],
            'TorchCREPE MIDI中位数': round(row['reference_median_pitch_midi'], 2),
            '误差(半音)': round(row['pitch_error_semitones'], 2),
            '疑似八度错误': row['is_octave_mismatch'],
            '有效帧数': row['voiced_frame_count'],
        })
rows.sort(key=lambda item: abs(item['误差(半音)']), reverse=True)
for row in rows[:15]:
    print(row)

print('\\n完整比较报告：', comparison_json)
print('没有重新运行任何模型；periodicity 阈值 0.5 仅作筛选，不代表人工真值。')
